# Phase 1 -> Phase 2: tokenize the corpora into training binariesRun this **once** on Kaggle with **CPU** (no GPU quota needed). It downloads thePhase 1 processed splits from Google Drive, encodes them with the Phase 1tokenizers, and writes compact `uint16` binaries for training to memory-map.Why binaries rather than training from the JSONL: the vocabulary is 8,000, so atoken id fits in 16 bits. 11.8 GB of JSONL becomes about 2.4 GB, training does noJSON parsing or tokenization in the data path, and the result can be saved as aKaggle Dataset that later sessions mount in seconds.### Disk budget`/kaggle/working` holds roughly **20 GB**. The Drive folder holds about 29 GB intotal, so this notebook downloads **only `Phase1-Processed`** (11.8 GB) anddeletes each JSONL as soon as it has been encoded. Peak usage stays near 12 GB.`Phase1-Manual` and `Phase1-Sources` are Phase 1 provenance and are not neededfor training.

In [ ]:
!pip install -q gdown sentencepieceimport os, json, time, shutil, subprocessfrom pathlib import Pathimport numpy as npWORK = Path("/kaggle/working")def disk():    st = os.statvfs(WORK)    free = st.f_bavail * st.f_frsize / 1e9    used = (st.f_blocks - st.f_bavail) * st.f_frsize / 1e9    print(f"disk: {used:.1f} GB used, {free:.1f} GB free")disk()

## 0. Clean any previous attemptIf a previous run filled the disk, this clears it. `/kaggle/working` persistswithin a session, so a failed attempt leaves its partial downloads behind.

In [ ]:
for p in ("corpora", "repo", "tokenized"):    shutil.rmtree(WORK / p, ignore_errors=True)disk()

## 1. Tokenizers from GitHubSmall files (about 200 KB each), so this is quick. They live in the repositoryrather than on Drive.

In [ ]:
REPO = "https://github.com/CL3-410/individual-project-Karan-thayat.git"TOK = WORK / "repo"# --depth 1 avoids pulling the whole history.r = subprocess.run(["git","clone","-q","--depth","1","--branch","phase-1",REPO,str(TOK)],                   capture_output=True, text=True)print("clone:", "ok" if r.returncode == 0 else r.stderr[:400])models = sorted(TOK.rglob("*_unigram_8000.model"))for m in models:    print(f"  {m.relative_to(TOK)}  {m.stat().st_size/1024:.0f} KB")assert len(models) == 2, f"expected 2 tokenizer models, found {len(models)}"

## 2. Download only the processed splits`Phase1-Processed` alone, not the whole Drive folder. If the folder downloadtruncates one of the large train files, use the per-file fallback below.

In [ ]:
PROCESSED_FOLDER = "https://drive.google.com/drive/folders/1ggiw-RzYTUgkXLhky2u73tObkw7MKbJT"RAW = WORK / "corpora"RAW.mkdir(parents=True, exist_ok=True)# Download the Phase1-Processed subfolder only. gdown --folder takes the folder# URL; point it at Phase1-Processed directly (open that subfolder in Drive and# copy its own link) to avoid pulling Phase1-Manual and Phase1-Sources too.!gdown --folder "{PROCESSED_FOLDER}" -O {RAW} --remaining-okfound = sorted(p for p in RAW.rglob("*.jsonl") if p.is_file())for p in found:    print(f"{p.stat().st_size/1e9:7.2f} GB  {p.relative_to(RAW)}")disk()

**If the above pulled more than `Phase1-Processed`** (you will see`nepali_mixed_300M_tokens.jsonl`, `indiccorp.txt`, or similar), stop, run thecleanup cell, and instead paste the link to the **Phase1-Processed subfolderitself** into `PROCESSED_FOLDER` above: open it in Drive, then Share -> Copylink.Expected sizes: `hi_train.jsonl` 5.13 GB, `ne_train.jsonl` 6.34 GB, and foursmall validation/test files. Anything materially smaller was truncated - use theper-file fallback:

In [ ]:
# Fallback: fetch one file by id (Drive -> right-click file -> Share -> Copy link,# the id sits between /d/ and /view).# !gdown --id PASTE_FILE_ID -O {RAW}/hi_train.jsonl

## 3. Encode, deleting each split as it is consumedDocuments are concatenated into one stream with an EOS token between them, thestandard arrangement for causal LM pretraining: document boundaries become anordinary token and training windows can start anywhere.Each JSONL is deleted immediately after encoding, which is what keeps peak diskusage near 12 GB rather than 14 GB.

In [ ]:
import sentencepiece as spmOUT = WORK / "tokenized"OUT.mkdir(exist_ok=True)CHUNK_DOCS = 20_000def locate(pattern: str, root: Path) -> Path:    hits = sorted(root.rglob(pattern))    if not hits:        raise FileNotFoundError(f"{pattern} not found under {root}")    return hits[0]def encode_split(jsonl_path: Path, sp, out_path: Path) -> dict:    """Encode one split to a flat uint16 array. Returns a manifest entry."""    eos = sp.eos_id()    assert sp.get_piece_size() <= 65535, "uint16 requires vocab <= 65535"    n_docs = n_tokens = 0    buf: list[int] = []    t0 = time.time()    with open(out_path, "wb") as out, open(jsonl_path, encoding="utf-8", errors="replace") as f:        for line in f:            try:                text = json.loads(line).get("text", "")            except Exception:                continue            if not text:                continue            ids = sp.encode(text)            buf.extend(ids); buf.append(eos)            n_docs += 1; n_tokens += len(ids) + 1            if n_docs % CHUNK_DOCS == 0:                np.asarray(buf, dtype=np.uint16).tofile(out); buf.clear()                rate = n_tokens / max(time.time() - t0, 1e-9)                print(f"   {n_docs:>9,} docs  {n_tokens/1e6:>8.1f}M tok  {rate/1e6:.2f}M tok/s", flush=True)        if buf:            np.asarray(buf, dtype=np.uint16).tofile(out)    return {"documents": n_docs, "tokens": n_tokens,            "bytes": out_path.stat().st_size, "seconds": round(time.time()-t0, 1)}manifest = {}for lang in ("hi", "ne"):    sp = spm.SentencePieceProcessor(model_file=str(locate(f"{lang}_unigram_8000.model", TOK)))    manifest[lang] = {"vocab_size": sp.get_piece_size(), "splits": {}}    for split in ("train", "validation", "test"):        src = locate(f"{lang}_{split}.jsonl", RAW)        print(f"\n[{lang}/{split}]  {src.stat().st_size/1e9:.2f} GB")        entry = encode_split(src, sp, OUT / f"{lang}_{split}.bin")        manifest[lang]["splits"][split] = entry        print(f"   {entry['tokens']:,} tokens -> {entry['bytes']/1e9:.2f} GB")        src.unlink()          # free the JSONL immediately        disk()(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))print("\nwrote manifest.json")

## 4. Verify before trusting itTwo checks worth doing now rather than three hours into a training run: tokencounts consistent with Phase 1, and decoded text that is readable and in theright language.

In [ ]:
EXPECTED = {"hi": 578_319_564, "ne": 610_989_716}   # Phase 1 reported train tokensfor lang, spec in manifest.items():    got, want = spec["splits"]["train"]["tokens"], EXPECTED[lang]    # The binary adds one EOS per document, so it runs slightly above Phase 1.    print(f"{lang}: {got:,} vs Phase 1 {want:,}  ({100*(got-want)/want:+.2f}%)")print()for lang in ("hi", "ne"):    sp = spm.SentencePieceProcessor(model_file=str(locate(f"{lang}_unigram_8000.model", TOK)))    arr = np.memmap(OUT / f"{lang}_train.bin", dtype=np.uint16, mode="r")    print(f"[{lang}] {len(arr):,} tokens on disk")    print("   ", sp.decode([int(t) for t in arr[10_000:10_060]])[:200], "\n")

## 5. Save as a Kaggle Dataset**Do not skip this.** `/kaggle/working` is wiped when the session ends; withoutsaving, the whole run is lost.Save the notebook version, then from the **Output** tab choose *Create Dataset*and name it `lma-phase2-tokenized`. Training sessions then mount it read-only at`/kaggle/input/lma-phase2-tokenized/`.

In [ ]:
shutil.rmtree(RAW, ignore_errors=True)shutil.rmtree(TOK, ignore_errors=True)total = 0for p in sorted(OUT.iterdir()):    total += p.stat().st_size    print(f"{p.stat().st_size/1e9:7.2f} GB  {p.name}")print(f"\n{total/1e9:.2f} GB total - save this as a Kaggle Dataset named lma-phase2-tokenized")disk()